# Blinded Stage-1 validation review

Review the held-out generation outputs without loading `review_mapping.jsonl`. The intended two-vector shortlist yields 208 tasks; a smaller eligible shortlist yields fewer. Every save atomically checkpoints to Google Drive. Set an immutable repository commit before running.

In [ ]:
REPOSITORY = 'https://github.com/ashioyajotham/safety_governor.git'
REVISION = 'SET_IMMUTABLE_COMMIT'
REVIEWER = 'Victor'
if REVISION == 'SET_IMMUTABLE_COMMIT':
    raise ValueError('Set REVISION to the immutable validation-workbench commit')
![ -d /content/safety_governor_validation/.git ] || git clone --quiet $REPOSITORY /content/safety_governor_validation
!git -C /content/safety_governor_validation fetch --quiet origin
!git -C /content/safety_governor_validation checkout --quiet $REVISION
%cd /content/safety_governor_validation
!python -m pip install --quiet -r requirements-review.txt

In [ ]:
from google.colab import drive, files
from pathlib import Path
drive.mount('/content/drive')
uploaded = files.upload()  # choose review_tasks.jsonl only; never review_mapping.jsonl
TASKS = Path('/content') / next(iter(uploaded))
DECISIONS = Path('/content/drive/MyDrive/safety_governor/validation_review_decisions.jsonl')
DECISIONS.parent.mkdir(parents=True, exist_ok=True)
print('Tasks:', TASKS)
print('Durable checkpoint:', DECISIONS)

In [ ]:
from safety_governor.validation_widgets import build_validation_review_widget
panel = build_validation_review_widget(TASKS, DECISIONS, reviewer=REVIEWER)

When the panel reports that every task is complete, download `validation_review_decisions.jsonl` from Drive and import it with `python -m scripts.validation_review summarize`. A runtime disconnect does not erase saved rows.